# Natural Language Processing: Transfer Learning with BERT
### Predicting Disaster Tweets using HuggingFace and PyTorch

**Objective:**
The goal of this notebook is to classify Twitter data to determine whether a tweet is reporting a genuine disaster or metaphorically using disaster vocabulary. 

**Methodology:**
Where traditional models (like TF-IDF or basic RNNs) struggle to capture grammatical nuance and long-term context, modern NLP relies on the "Attention Mechanism." In this notebook, we utilize **Transfer Learning**. We import **DistilBERT**, a Transformer architecture pre-trained on the English Wikipedia corpus, and fine-tune its classification head exclusively on the Kaggle tweet set using native PyTorch.


In [ ]:
import pandas as pd
import numpy as np
import torch
import warnings
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore")

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Apple Silicon MPS GPU is locked and loaded!")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Nvidia CUDA GPU is active!")
else:
    device = torch.device("cpu")
    print("Running on CPU.")

---
## Chapter 1: Data Ingestion
Twitter text is inherently noisy. Before feeding data into the Transformer, we load the raw strings. Unlike classical NLP approaches, we intentionally **skip aggressive text cleaning** (e.g., removing stop-words or lemmatization). Transformers map semantic relationships using the literal grammar of the sentence; prematurely deleting stop-words destroys the attention-context the model relies on.


In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/test.csv")

X_train_raw = train_df['text'].values
y_train = train_df['target'].values
X_test_raw = test_df['text'].values

---
## Chapter 2: The BERT Tokenizer
Pre-trained Transformers utilize massive, immutable vocabularies. To ensure our data aligns with the model's pre-existing knowledge vectors, we must use the exact Tokenizer designed for our specific BERT checkpoint. 

The tokenizer will convert our raw text into integer IDs, while automatically generating an `attention_mask`. This mask explicitly instructs the AI which tokens contain semantic meaning, and which represent empty padding to be ignored during training.


In [ ]:
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

train_encodings = tokenizer(list(X_train_raw), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test_raw), truncation=True, padding=True, max_length=128)

---
## Chapter 3: PyTorch Data Handling
For maximum computational efficiency during GPU training, PyTorch requires data to be formally wrapped in a `torch.utils.data.Dataset` subclass. 

This strict object-oriented approach explicitly defines how tensor batches are extracted and shipped from RAM to the GPU. We will define a custom `TweetDataset` class to package our tokenized inputs and target labels perfectly for the HuggingFace Trainer.


In [ ]:
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = TweetDataset(train_encodings, y_train)
test_dataset = TweetDataset(test_encodings)


---
## Chapter 4: Transfer Learning (Model Instantiation)
We now securely download the pre-trained `DistilBERT` architecture. By calling `ForSequenceClassification`, the HuggingFace library automatically removes BERT's original output layer (which was designed for masked language modeling) and attaches a completely blank, randomized 2-neuron classification head (Disaster vs. Safe). 

Our training loop will essentially "freeze" the embedded Wikipedia knowledge and purely train this new head to map that vocabulary to our Kaggle objective.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, 
    num_labels=2 
)

model.to(device)

print(f"DistilBERT successfully loaded onto {device.type.upper()}!") 

---
## Chapter 5: Advanced Fine-Tuning (HuggingFace Trainer)
Because we are modifying a pre-trained architectural masterpiece, standard Deep Learning hyperparameters would destroy the model. Using a standard learning rate (e.g., `1e-3`) would cause violent mathematical updates that shatter BERT's understanding of English. 

Instead, we use a microscopic learning rate (`2e-5`) to gently "nudge" the weights. The HuggingFace `Trainer` API abstracts away the complex PyTorch training loop, handling evaluation and optimization under the hood.


In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    learning_rate=2e-5,             
    weight_decay=0.01,            
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics
)

print("Commencing Deep Transfer Learning...")
trainer.train()

---
## Chapter 6: Inference and Kaggle Export
With the classification head successfully tuned to the training set, we execute inference on the unknown test data. HuggingFace naturally outputs `logits` (raw mathematical distances). We apply an `argmax` function to determine which of the two neurons fired harder for every individual tweet, yielding our final binary Kaggle predictions.


In [ ]:
print("Running BERT Inference on Test Data...")
predictions_output = trainer.predict(test_dataset)

bert_predictions = np.argmax(predictions_output.predictions, axis=1)

submission_bert = pd.DataFrame({
    "id": test_df["id"],
    "target": bert_predictions
})

submission_bert.to_csv("submission.csv", index=False)
print("BERT Submission saved to 'bert_submission.csv'!")